# Modelo de Previsão do Índice de Ilha de Calor Urbana (UHI)

Este notebook demonstra o processo de desenvolvimento de um modelo para prever o Índice de Ilha de Calor Urbana (UHI) utilizando dados de satélite como features, sem usar coordenadas geográficas como preditores.


## 1. Configuração do Ambiente e Bibliotecas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configurações visuais para gráficos mais legíveis
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
warnings.filterwarnings('ignore')

# Configurações para melhor visualização dos dataframes
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 2. Carregamento e Exploração dos Dados

Vamos carregar o conjunto de dados de treinamento e explorar sua estrutura:

In [ ]:
# Carregar os dados de treinamento
df_train = pd.read_csv('dataset/training_data.csv')

# Exibir as primeiras linhas
print("Primeiras linhas do conjunto de dados:")
df_train.head()

In [ ]:
# Informações sobre as colunas
print("\nInformações sobre o conjunto de dados:")
df_train.info()

In [ ]:
# Estatísticas descritivas
print("\nEstatísticas descritivas:")
df_train.describe()

## 3. Pré-processamento dos Dados
Realizando as transformações iniciais nos dados para prepará-los para análise:


In [ ]:
# Remover a coluna de timestamp
df_train = df_train.drop('datetime', axis=1)

# Verificar as colunas restantes
print("Colunas após remoção do timestamp:")
df_train.columns

In [ ]:
# Verificar valores nulos
print("\nValores nulos por coluna:")
df_train.isnull().sum()

## 4. Análise Exploratória dos Dados (EDA)

Explorando a distribuição do Índice UHI e sua relação espacial para entender os padrões subjacentes:


In [ ]:
# Histograma do Índice UHI
plt.figure(figsize=(10, 6))
sns.histplot(df_train['UHI Index'], kde=True)
plt.title('Distribuição do Índice de Ilha de Calor Urbana (UHI)')
plt.xlabel('Índice UHI')
plt.ylabel('Frequência')
plt.grid(True)
plt.show()

In [ ]:
# Scatterplot das localizações coloridas pelo Índice UHI
plt.figure(figsize=(12, 10))
scatter = plt.scatter(df_train['Longitude'], df_train['Latitude'], 
                    c=df_train['UHI Index'], cmap='coolwarm', 
                    alpha=0.6, s=10)
plt.colorbar(scatter, label='Índice UHI')
plt.title('Distribuição Espacial do Índice UHI')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True)
plt.show()

In [ ]:
# Estatísticas por quartis do Índice UHI
print("Estatísticas por quartis do Índice UHI:")
df_train['UHI_quartil'] = pd.qcut(df_train['UHI Index'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
df_train.groupby('UHI_quartil')['UHI Index'].describe()

## 5. Extração de Dados de Satélite

Configurando o ambiente para extrair dados de satélite do Microsoft Planetary Computer, focando nos dados do Sentinel-2 e Landsat que correspondam ao período de estudo:


In [ ]:
import planetary_computer
from pystac_client import Client
from odc.stac import stac_load
from datetime import datetime, timedelta
import pystac_client

# Verificar a versão do stackstac para debugging
import importlib.metadata
print(f"Versão do stackstac: {importlib.metadata.version('stackstac')}")

In [ ]:
# Inicializar o cliente Planetary Computer
pc = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

# Definir intervalo de tempo para busca de imagens
# Podemos buscar imagens alguns dias antes da coleta para garantir cobertura de nuvens mínima
data_coleta = "2021-07-24"
data_inicio = (datetime.strptime(data_coleta, "%Y-%m-%d") - timedelta(days=30)).strftime("%Y-%m-%d")
data_fim = (datetime.strptime(data_coleta, "%Y-%m-%d") + timedelta(days=30)).strftime("%Y-%m-%d")

# Extrair coordenadas para definir a área de busca
min_lon = df_train['Longitude'].min()
max_lon = df_train['Longitude'].max()
min_lat = df_train['Latitude'].min()
max_lat = df_train['Latitude'].max()

# Definir o bounding box (bbox) para a área de estudo
bbox = [min_lon, min_lat, max_lon, max_lat]

print(f"Período de busca: {data_inicio} a {data_fim}")
print(f"Área de estudo (bbox): {bbox}")

### 5.1 Extração e Processamento de Dados do Sentinel-2
Acessando e processando imagens do Sentinel-2 para extrair informações espectrais e índices relevantes:


In [ ]:
# Buscar coleção Sentinel-2 Level 2A (reflectância de superfície)
search_sentinel = pc.search(
    collections=["sentinel-2-l2a"],
    datetime=f"{data_inicio}/{data_fim}",
    bbox=bbox,
    query={"eo:cloud_cover": {"lt": 10}}  # Filtrar imagens com menos de 10% de cobertura de nuvens
)

# Verificar número de cenas encontradas
sentinel_items = list(search_sentinel.get_items())
print(f"Número de cenas Sentinel-2 encontradas: {len(sentinel_items)}")

# Visualizar metadados da primeira cena para verificação
if len(sentinel_items) > 0:
    print("Metadados da primeira cena Sentinel-2:")
    print(f"ID: {sentinel_items[0].id}")
    print(f"Data: {sentinel_items[0].properties['datetime']}")
    print(f"Cobertura de nuvens: {sentinel_items[0].properties['eo:cloud_cover']}%")

In [ ]:
# Define the pixel resolution for the final product
# Define the scale according to our selected crs, so we will use degrees
resolution = 10  # meters per pixel 
scale = resolution / 111320.0 # degrees per pixel for crs=4326 

In [ ]:
# Carregar dados do Sentinel-2 com as bandas relevantes
data = stac_load(
    sentinel_items,
    bands=["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"],
    crs="EPSG:4326",  # Sistema de coordenadas em Latitude-Longitude
    resolution=scale,  # Resolução espacial em graus
    chunks={"x": 2048, "y": 2048},
    dtype="uint16",
    patch_url=planetary_computer.sign,
    bbox=bbox
)

In [ ]:
# Visualizar imagens da série temporal
plot_data = data[["B04","B03","B02"]].to_array()
plot_data.plot.imshow(col='time', col_wrap=4, robust=True, vmin=0, vmax=2500)
plt.show() 

In [ ]:
# Visualizar uma imagem RGB de uma única data
fig, ax = plt.subplots(figsize=(6,6))
plot_data.isel(time=0).plot.imshow(robust=True, ax=ax, vmin=0, vmax=2500)
ax.axis('off')
plt.show()

In [ ]:
# Criar composição mediana para remover nuvens e outros ruídos temporais
median = data.median(dim="time").compute()

# Visualizar a composição RGB mediana
fig, ax = plt.subplots(figsize=(6,6))
median[["B04", "B03", "B02"]].to_array().plot.imshow(robust=True, ax=ax, vmin=0, vmax=2500)
ax.set_title("Composição RGB Mediana")
ax.axis('off')
plt.show()

In [ ]:
# Calcular e visualizar o NDVI (Índice de Vegetação por Diferença Normalizada)
ndvi_median = (median.B08-median.B04)/(median.B08+median.B04)
fig, ax = plt.subplots(figsize=(7,6))
ndvi_median.plot.imshow(vmin=0.0, vmax=1.0, cmap="RdYlGn")
plt.title("NDVI Mediano")
plt.axis('off')
plt.show()

In [ ]:
# Calcular e visualizar o NDBI (Índice de Construção por Diferença Normalizada)
ndbi_median = (median.B11-median.B08)/(median.B11+median.B08)
fig, ax = plt.subplots(figsize=(7,6))
ndbi_median.plot.imshow(vmin=-0.1, vmax=0.1, cmap="jet")
plt.title("NDBI Mediano")
plt.axis('off')
plt.show()

In [ ]:
# Calcular e visualizar o NDWI (Índice de Água por Diferença Normalizada)
ndwi_median = (median.B03-median.B08)/(median.B03+median.B08)
fig, ax = plt.subplots(figsize=(7,6))
ndwi_median.plot.imshow(vmin=-0.3, vmax=0.3, cmap="RdBu")
plt.title("NDWI Mediano")
plt.axis('off')
plt.show()

### 5.3 Extração de Dados do Landsat
Obtendo e processando imagens do Landsat 8 para extrair informações térmicas e índices adicionais:


In [ ]:
# Inicializar cliente STAC para acesso ao Landsat
stac = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

# Definir intervalo de tempo mais amplo para Landsat devido à menor frequência temporal
data_coleta = "2021-07-24"
data_inicio = (datetime.strptime(data_coleta, "%Y-%m-%d") - timedelta(days=60)).strftime("%Y-%m-%d")
data_fim = (datetime.strptime(data_coleta, "%Y-%m-%d") + timedelta(days=60)).strftime("%Y-%m-%d")

# Buscar cenas do Landsat 8
landsat_items = stac.search(
    bbox=bbox, 
    datetime=f"{data_inicio}/{data_fim}",
    collections=["landsat-c2-l2"],
    query={"eo:cloud_cover": {"lt": 20}, "platform": {"in": ["landsat-8"]}},
)

# Verificar número de cenas encontradas
landsat_items = list(landsat_items.get_items())
print(f"Número de cenas Landsat 8 encontradas: {len(landsat_items)}")

# Visualizar metadados da primeira cena para verificação
if len(landsat_items) > 0:
    print("Metadados da primeira cena Landsat:")
    print(f"ID: {landsat_items[0].id}")
    print(f"Data: {landsat_items[0].properties['datetime']}")
    print(f"Cobertura de nuvens: {landsat_items[0].properties['eo:cloud_cover']}%")

In [ ]:
# Carregar bandas RGB e NIR do Landsat
data1 = stac_load(
    landsat_items,
    bands=["red", "green", "blue", "nir08"],
    crs="EPSG:4326",
    resolution=scale,
    chunks={"x": 2048, "y": 2048},
    dtype="uint16",
    patch_url=planetary_computer.sign,
    bbox=bbox
)

In [ ]:
# Carregar banda térmica do Landsat
data2 = stac_load(
    landsat_items,
    bands=["lwir11"],  # Banda de infravermelho térmico
    crs="EPSG:4326",
    resolution=scale,
    chunks={"x": 2048, "y": 2048},
    dtype="uint16",
    patch_url=planetary_computer.sign,
    bbox=bbox
)

In [ ]:
# Persistir os dados em memória para operações mais rápidas
data1 = data1.persist()
data2 = data2.persist()

In [ ]:
# Aplicar fatores de escala para as bandas RGB e NIR
scale1 = 0.0000275 
offset1 = -0.2 
data1 = data1.astype(float) * scale1 + offset1

# Aplicar fatores de escala para a banda de temperatura de superfície
# e converter de Kelvin para Celsius
scale2 = 0.00341802 
offset2 = 149.0 
kelvin_celsius = 273.15
data2 = data2.astype(float) * scale2 + offset2 - kelvin_celsius

In [ ]:
# Visualizar série temporal RGB
plot_data = data1[["red","green","blue"]].to_array()
plot_data.plot.imshow(col='time', col_wrap=4, robust=True, vmin=0, vmax=0.25)
plt.show()

In [ ]:
# Pick one of the scenes above (numbering starts with 0)
scene = 0

In [ ]:
# Visualizar imagem RGB de cor real para a cena selecionada
fig, ax = plt.subplots(figsize=(9,10))
data1.isel(time=scene)[["red", "green", "blue"]].to_array().plot.imshow(robust=True, ax=ax, vmin=0.0, vmax=0.25)
ax.set_title("Imagem RGB de Cor Real")
ax.axis('off')
plt.show()

In [ ]:
# Calcular e visualizar o NDVI para a cena selecionada
ndvi_data = (data1.isel(time=scene).nir08-data1.isel(time=scene).red)/(data1.isel(time=scene).nir08+data1.isel(time=scene).red)
fig, ax = plt.subplots(figsize=(11,10))
ndvi_data.plot.imshow(vmin=0.0, vmax=1.0, cmap="RdYlGn")
plt.title("Índice de Vegetação (NDVI) - Landsat")
plt.axis('off')
plt.show()

In [ ]:
# Visualizar a temperatura de superfície terrestre (LST)
fig, ax = plt.subplots(figsize=(11,10))
data2.isel(time=scene).lwir11.plot.imshow(vmin=20.0, vmax=45.0, cmap="jet")
plt.title("Temperatura de Superfície Terrestre (LST)")
plt.axis('off')
plt.show()

## 6. Extração de Features para o Conjunto de Treinamento
Definindo funções para extração de valores de pixels em coordenadas específicas e aplicando-as aos pontos de treinamento:


In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
from tqdm import tqdm

In [ ]:
# Função para extrair valores em coordenadas específicas
def extract_pixel_values(dataset, lon, lat):
    """
    Extrai o valor do pixel mais próximo das coordenadas fornecidas.
    
    Parâmetros:
    -----------
    dataset : xarray.Dataset ou xarray.DataArray
        Dataset ou DataArray do qual extrair valores
    lon : float
        Longitude do ponto
    lat : float
        Latitude do ponto
        
    Retorna:
    --------
    float ou dict
        Valor do pixel na coordenada especificada ou
        dicionário com valores para cada banda
    """
    try:
        if isinstance(dataset, xr.Dataset):
            # Se for um Dataset, retorna um dicionário com valores para cada banda
            result = {}
            for var in dataset.data_vars:
                try:
                    point_data = dataset[var].sel(longitude=lon, latitude=lat, method="nearest")
                    
                    if "time" in point_data.dims:
                        result[var] = float(point_data.median().values)
                    else:
                        if hasattr(point_data, 'values'):
                            result[var] = float(point_data.values)
                        else:
                            result[var] = float(point_data)
                except Exception as e:
                    print(f"Erro ao processar variável {var}: {e}")
                    result[var] = np.nan
            return result
        else:
            # Se for um DataArray, retorna o valor diretamente
            try:
                point_data = dataset.sel(longitude=lon, latitude=lat, method="nearest")
                
                if "time" in point_data.dims:
                    return float(point_data.median().values)
                else:
                    if hasattr(point_data, 'values'):
                        return float(point_data.values)
                    else:
                        return float(point_data)
            except Exception as e:
                print(f"Erro ao processar DataArray: {e}")
                return np.nan
    except Exception as e:
        print(f"Erro geral na extração: {e}")
        return np.nan

In [ ]:
# Extrair valores para todos os pontos do conjunto de treinamento
print("Extraindo índices dos satélites para pontos de treinamento...")

# Verificar estrutura dos datasets para debugging
print("Dimensões do dataset Sentinel-2 (median):")
print(median.dims)
print("\nVariáveis do dataset Sentinel-2:")
print(list(median.data_vars))

print("\nDimensões do dataset NDVI Sentinel-2:")
print(ndvi_median.dims)

print("\nDimensões do dataset Landsat (data1):")
try:
    print(data1.dims)
    print("\nVariáveis do dataset Landsat:")
    print(list(data1.data_vars))
except Exception as e:
    print(f"Erro ao acessar informações do Landsat: {e}")

In [ ]:
# Lista para armazenar features de cada ponto
feature_data = []

# Para cada ponto no conjunto de treinamento
for index, row in tqdm(df_train.iterrows(), total=len(df_train)):
    lon, lat = row['Longitude'], row['Latitude']
    
    # Dicionário para armazenar features deste ponto
    point_features = {
        'id': index,
        'Longitude': lon,
        'Latitude': lat,
        'UHI Index': row['UHI Index']
    }
    
    # Extrair bandas do Sentinel-2 com tratamento de erros
    try:
        # Extrair valores das bandas
        bands = extract_pixel_values(median, lon, lat)
        if isinstance(bands, dict):
            for band, value in bands.items():
                point_features[f'S2_{band}'] = value
        else:
            print(f"Resultado inesperado para bandas S2 no ponto {index}: {type(bands)}")
            # Preencher com NaN se bands não for um dicionário
            for band in median.data_vars:
                point_features[f'S2_{band}'] = np.nan
                
        # Extrair índices calculados do Sentinel-2
        try:
            ndvi_value = ndvi_median.sel(longitude=lon, latitude=lat, method="nearest").values
            point_features['NDVI_S2'] = float(ndvi_value)
        except Exception as e:
            print(f"Erro ao extrair NDVI_S2 para ponto {index}: {e}")
            point_features['NDVI_S2'] = np.nan
            
        try:
            ndbi_value = ndbi_median.sel(longitude=lon, latitude=lat, method="nearest").values
            point_features['NDBI_S2'] = float(ndbi_value)
        except Exception as e:
            print(f"Erro ao extrair NDBI_S2 para ponto {index}: {e}")
            point_features['NDBI_S2'] = np.nan
            
        try:
            ndwi_value = ndwi_median.sel(longitude=lon, latitude=lat, method="nearest").values
            point_features['NDWI_S2'] = float(ndwi_value)
        except Exception as e:
            print(f"Erro ao extrair NDWI_S2 para ponto {index}: {e}")
            point_features['NDWI_S2'] = np.nan
        
    except Exception as e:
        print(f"Erro ao extrair dados Sentinel-2 para ponto {index}: {e}")
        # Preencher com NaN em caso de erro
        try:
            for band in median.data_vars:
                point_features[f'S2_{band}'] = np.nan
        except:
            print("Erro ao acessar data_vars do Sentinel-2")
        
        point_features['NDVI_S2'] = np.nan
        point_features['NDBI_S2'] = np.nan
        point_features['NDWI_S2'] = np.nan
    
    # Extrair dados do Landsat com tratamento de erros
    try:
        # NDVI do Landsat - acesso direto usando dimensões conhecidas
        landsat_point = data1.isel(time=scene)
        
        # Extrai NIR e Red com acesso direto
        try:
            nir = landsat_point.nir08.sel(longitude=lon, latitude=lat, method="nearest").values
            nir = float(nir)
        except Exception as e:
            print(f"Erro ao extrair NIR para ponto {index}: {e}")
            nir = np.nan
            
        try:
            red = landsat_point.red.sel(longitude=lon, latitude=lat, method="nearest").values
            red = float(red)
        except Exception as e:
            print(f"Erro ao extrair RED para ponto {index}: {e}")
            red = np.nan
        
        # Calcula NDVI se possível
        if not (np.isnan(nir) or np.isnan(red)):
            ndvi_landsat = (nir - red) / (nir + red) if (nir + red) != 0 else np.nan
        else:
            ndvi_landsat = np.nan
            
        point_features['NDVI_LS'] = ndvi_landsat
        
        # LST (Temperatura de Superfície) do Landsat
        try:
            lst = data2.isel(time=scene).lwir11.sel(longitude=lon, latitude=lat, method="nearest").values
            point_features['LST_LS'] = float(lst)
        except Exception as e:
            print(f"Erro ao extrair LST para ponto {index}: {e}")
            point_features['LST_LS'] = np.nan
            
    except Exception as e:
        print(f"Erro ao extrair dados Landsat para ponto {index}: {e}")
        point_features['NDVI_LS'] = np.nan
        point_features['LST_LS'] = np.nan
    
    feature_data.append(point_features)

In [ ]:
# Criar DataFrame com todas as features extraídas
df_features = pd.DataFrame(feature_data)

# Verificar valores ausentes
print("\nVerificando valores ausentes nas features extraídas:")
print(df_features.isnull().sum())

# Salvar o conjunto de dados completo com features
df_features.to_csv('dataset/training_with_satellite_features.csv', index=False)
print("\nDataset completo salvo em 'dataset/training_with_satellite_features.csv'")

# Visualizar as primeiras linhas do dataset com as features
print("\nPrimeiras linhas do dataset com features:")
df_features.head()

# Calcular correlações entre índices e UHI
print("\nCorrelações entre índices de satélite e UHI Index:")
correlations = df_features[[
    'UHI Index', 'NDVI_S2', 'NDBI_S2', 'NDWI_S2', 'NDVI_LS', 'LST_LS'
]].corr()['UHI Index'].sort_values(ascending=False)
print(correlations)

## 7. Extração de Features para Predição

Aplicando o mesmo processo de extração de features aos pontos do conjunto de teste para posterior predição:


In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
from tqdm import tqdm

In [ ]:
# Carregar o template de submissão
submission_df = pd.read_csv('dataset/submission_template.csv')
print(f"Arquivo de submissão carregado com {submission_df.shape[0]} pontos")
print(submission_df)

# Função otimizada para extrair valores em coordenadas específicas
def extract_pixel_values_at_coords(dataset, lon, lat):
    try:
        if isinstance(dataset, xr.Dataset):
            # Se for um Dataset, retorna um dicionário com valores para cada banda
            result = {}
            for var in dataset.data_vars:
                try:
                    point_data = dataset[var].sel(longitude=lon, latitude=lat, method="nearest")
                    
                    if "time" in point_data.dims:
                        result[var] = float(point_data.median().values)
                    else:
                        if hasattr(point_data, 'values'):
                            result[var] = float(point_data.values)
                        else:
                            result[var] = float(point_data)
                except Exception as e:
                    print(f"Erro ao processar variável {var}: {e}")
                    result[var] = np.nan
            return result
        else:
            # Se for um DataArray, retorna o valor diretamente
            try:
                point_data = dataset.sel(longitude=lon, latitude=lat, method="nearest")
                
                if "time" in point_data.dims:
                    return float(point_data.median().values)
                else:
                    if hasattr(point_data, 'values'):
                        return float(point_data.values)
                    else:
                        return float(point_data)
            except Exception as e:
                print(f"Erro ao processar DataArray: {e}")
                return np.nan
    except Exception as e:
        print(f"Erro geral na extração: {e}")
        return np.nan



In [ ]:
# Extrair features para cada ponto do arquivo de submissão
print("Extraindo índices e variáveis para os pontos de submissão...")

feature_data = []

# Para cada ponto no conjunto de submissão
for index, row in tqdm(submission_df.iterrows(), total=len(submission_df)):
    lon, lat = row['Longitude'], row['Latitude']
    
    # Dicionário para armazenar features deste ponto
    point_features = {
        'Longitude': lon,
        'Latitude': lat
    }
    
    # Extrair bandas do Sentinel-2
    try:
        bands = extract_pixel_values_at_coords(median, lon, lat)
        if isinstance(bands, dict):
            for band, value in bands.items():
                point_features[f'S2_{band}'] = value
        else:
            print(f"Resultado inesperado para bandas S2 no ponto {index}: {type(bands)}")
            
        # Extrair índices do Sentinel-2
        try:
            ndvi_value = ndvi_median.sel(longitude=lon, latitude=lat, method="nearest").values
            point_features['NDVI_S2'] = float(ndvi_value)
        except Exception as e:
            print(f"Erro ao extrair NDVI_S2 para ponto {index}: {e}")
            point_features['NDVI_S2'] = np.nan
            
        try:
            ndbi_value = ndbi_median.sel(longitude=lon, latitude=lat, method="nearest").values
            point_features['NDBI_S2'] = float(ndbi_value)
        except Exception as e:
            print(f"Erro ao extrair NDBI_S2 para ponto {index}: {e}")
            point_features['NDBI_S2'] = np.nan
            
        try:
            ndwi_value = ndwi_median.sel(longitude=lon, latitude=lat, method="nearest").values
            point_features['NDWI_S2'] = float(ndwi_value)
        except Exception as e:
            print(f"Erro ao extrair NDWI_S2 para ponto {index}: {e}")
            point_features['NDWI_S2'] = np.nan
            
    except Exception as e:
        print(f"Erro ao extrair dados Sentinel-2 para ponto {index}: {e}")
        # Preencher com NaN em caso de erro
        try:
            for band in median.data_vars:
                point_features[f'S2_{band}'] = np.nan
        except:
            print("Erro ao acessar data_vars do Sentinel-2")
        
        point_features['NDVI_S2'] = np.nan
        point_features['NDBI_S2'] = np.nan
        point_features['NDWI_S2'] = np.nan
    
    # Extrair dados do Landsat
    try:
        # NDVI do Landsat - acesso direto usando dimensões conhecidas
        landsat_point = data1.isel(time=scene)
        
        # Extrai NIR e Red com acesso direto
        try:
            nir = landsat_point.nir08.sel(longitude=lon, latitude=lat, method="nearest").values
            nir = float(nir)
        except Exception as e:
            print(f"Erro ao extrair NIR para ponto {index}: {e}")
            nir = np.nan
            
        try:
            red = landsat_point.red.sel(longitude=lon, latitude=lat, method="nearest").values
            red = float(red)
        except Exception as e:
            print(f"Erro ao extrair RED para ponto {index}: {e}")
            red = np.nan
        
        # Calcula NDVI se possível
        if not (np.isnan(nir) or np.isnan(red)):
            ndvi_landsat = (nir - red) / (nir + red) if (nir + red) != 0 else np.nan
        else:
            ndvi_landsat = np.nan
            
        point_features['NDVI_LS'] = ndvi_landsat
        
        # LST (Temperatura de Superfície) do Landsat
        try:
            lst = data2.isel(time=scene).lwir11.sel(longitude=lon, latitude=lat, method="nearest").values
            point_features['LST_LS'] = float(lst)
        except Exception as e:
            print(f"Erro ao extrair LST para ponto {index}: {e}")
            point_features['LST_LS'] = np.nan
            
    except Exception as e:
        print(f"Erro ao extrair dados Landsat para ponto {index}: {e}")
        point_features['NDVI_LS'] = np.nan
        point_features['LST_LS'] = np.nan
    
    feature_data.append(point_features)

# Criar DataFrame com todas as features extraídas
submission_features = pd.DataFrame(feature_data)

In [ ]:
# Verificar quaisquer valores ausentes
print("\nVerificando valores ausentes nas features extraídas:")
print(submission_features.isnull().sum())

# Salvar as features extraídas
submission_features.to_csv('dataset/submission_features.csv', index=False)
print("\nFeatures de submissão salvas em 'dataset/submission_features.csv'")

# Exibir as primeiras linhas das features extraídas
print("\nPrimeiras linhas das features extraídas:")
print(submission_features.head())